# 00 — PanTS Metadata Exploration
**AI-Based Pancreatic Cancer Prediction Using Deep Learning** · Phase-II Review 1

PanTS (Li et al., NeurIPS 2025) is the project's target dataset: 36,390 CT scans, 145 centres. The full image data is ~300 GB and is being staged on the college GPU machine for Review 2.
The **metadata file (1.3 MB)** ships with the Hugging Face mirror and already answers the questions that motivate this project: how many tumors are sub-2 cm, which contrast phases are present, and how cases are spread across hospitals.

Runs on CPU. Works in Colab or locally.

In [ ]:
!pip -q install pandas openpyxl matplotlib huggingface_hub
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
from huggingface_hub import hf_hub_download
path = hf_hub_download(repo_id='BodyMaps/PanTSMini', filename='metadata.xlsx', repo_type='dataset')
xl = pd.ExcelFile(path); print('sheets:', xl.sheet_names)
df = pd.concat([xl.parse(s).assign(sheet=s) for s in xl.sheet_names], ignore_index=True)
print(df.shape); df.columns.tolist()

The column names below come straight from the file — inspect them first, then set the four variables in the next cell to the matching columns. (Do not guess; print `df.head()` if unsure.)

In [ ]:
df.head()

In [ ]:
# ---- map the real column names here after inspecting df.columns ----
COL_SIZE   = next((c for c in df.columns if 'size' in c.lower() or 'diam' in c.lower()), None)
COL_PHASE  = next((c for c in df.columns if 'phase' in c.lower()), None)
COL_CENTER = next((c for c in df.columns if 'center' in c.lower() or 'hospital' in c.lower() or 'site' in c.lower()), None)
COL_DIAG   = next((c for c in df.columns if 'diag' in c.lower() or 'tumor' in c.lower()), None)
print(dict(size=COL_SIZE, phase=COL_PHASE, center=COL_CENTER, diagnosis=COL_DIAG))

In [ ]:
os.makedirs('results', exist_ok=True)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

if COL_SIZE:
    s = pd.to_numeric(df[COL_SIZE], errors='coerce').dropna()
    if s.max() > 30: s = s / 10.0      # mm -> cm if needed
    s.plot.hist(bins=40, ax=ax[0], color='#5b4fbf'); ax[0].axvline(2.0, color='r', ls='--')
    ax[0].set_title(f'Tumor size (cm) — sub-2 cm: {(s<2).sum()} / {len(s)} ({100*(s<2).mean():.1f}%)'); ax[0].set_xlabel('cm')
else: ax[0].set_title('size column not found')

if COL_PHASE:
    df[COL_PHASE].value_counts().plot.bar(ax=ax[1], color='#5b4fbf'); ax[1].set_title('Contrast phase')
else: ax[1].set_title('phase column not found')

if COL_CENTER:
    vc = df[COL_CENTER].value_counts()
    vc.head(20).plot.bar(ax=ax[2], color='#5b4fbf'); ax[2].set_title(f'Top 20 of {vc.size} centres (scans per centre)')
else: ax[2].set_title('centre column not found')

plt.tight_layout(); plt.savefig('results/fig_pants_metadata.png', dpi=150); plt.show()

In [ ]:
summary = {
    'total_rows': int(len(df)),
    'sheets': xl.sheet_names,
    'n_centres': int(df[COL_CENTER].nunique()) if COL_CENTER else None,
    'phases': df[COL_PHASE].value_counts().to_dict() if COL_PHASE else None,
    'diagnosis': df[COL_DIAG].value_counts().head(15).to_dict() if COL_DIAG else None,
}
import json; print(json.dumps(summary, indent=1, default=str))
json.dump(summary, open('results/pants_metadata_summary.json','w'), indent=1, default=str)

**For the deck:** one slide — "PanTS at a glance" — with `fig_pants_metadata.png` and the sub-2 cm percentage. It ties the dataset chapter of the report to a number the panel can remember, and it shows real engagement with PanTS before the images are on disk.